# AirfRANS GNN Surrogate — Kaggle GPU setup

Fallback for when Colab's free GPU quota runs out -- separate quota pool
(~30 hrs/week of P100 or T4x2). No Drive-equivalent live mount here: Kaggle
persists across sessions via **Datasets** (read-only, attached to a session)
and a notebook's own **Output** (from "Save Version"), not a synced folder.

Before running: Settings (right sidebar) > Accelerator > GPU, and
Internet > On (needed for git clone / pip install / dataset download).

## 0. Resume from the Colab run

Colab's session got 25 epochs in before hitting the GPU limit, with periodic
checkpoints saved to Drive. Downloaded `mgn-epoch=024.ckpt` from Drive and
uploaded it here as a private Kaggle Dataset (Add Data > Upload) -- fill in
the path Kaggle assigned it below. New checkpoints from this session save to
`/kaggle/working` instead (Kaggle's own persistence, see section 6).

In [ ]:
# Replace with wherever Kaggle mounted your uploaded checkpoint, e.g.
# "/kaggle/input/mgn-checkpoint/mgn-epoch=024.ckpt"
RESUME_CHECKPOINT = "/kaggle/input/<your-dataset-name>/mgn-epoch=024.ckpt"

import os
print("found:", os.path.isfile(RESUME_CHECKPOINT))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo and install dependencies

Same as Colab -- Kaggle also ships CUDA-enabled torch preinstalled, and
`torch_geometric` installs as pure Python (no `torch-scatter`/`torch-sparse`
needed, confirmed working on both machines already).

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/airfrans-gnn-surrogate.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q torch_geometric lightning airfrans pyvista

## 2. Get the dataset

Redownloaded fresh here rather than transferring the ~9.6GB cache over from
Drive (that's a bigger one-time hop -- revisit if this becomes a recurring
pattern). `manifest.json` doesn't need transferring either way, it's
committed to the repo at `data/manifest.json`.

In [ ]:
import airfrans as af

DATA_ROOT = "data"
dataset_root = os.path.join(DATA_ROOT, "Dataset")

if not os.path.isdir(dataset_root) or not os.listdir(dataset_root):
    af.dataset.download(root=DATA_ROOT, file_name="Dataset", unzip=True, OpenFOAM=False)

print("manifest present:", os.path.isfile(os.path.join(dataset_root, "manifest.json")))

## 3. Preprocess: cache real mesh graphs

Writes to `/kaggle/working` (this session's local disk) -- one-time cost per
case, same as the Colab run. Deletes the raw dataset afterward, same reasoning
as Colab: training only reads the cache, and freeing the space is cheap
insurance against disk limits regardless of Kaggle's exact quota.

In [ ]:
import shutil

from src.data import split_names
from src.preprocess import preprocess_split

CACHE_DIR = "/kaggle/working/cache/full"
MANIFEST_DIR = "data"  # data/manifest.json is committed to the repo -- always present
train_names = split_names(dataset_root, task="full", train=True)

preprocess_split(dataset_root, train_names, CACHE_DIR)
shutil.rmtree(dataset_root, ignore_errors=True)
print(f"cached {len(os.listdir(CACHE_DIR))}/{len(train_names)} training cases")

## 4. Train, resuming from the Colab checkpoint

`resume_from_checkpoint` is passed explicitly here (pointing at the uploaded
Dataset file from section 0) rather than left to auto-detect, since this
session's own `checkpoint_dir` starts empty -- auto-detect only finds
checkpoints saved *within* the current run. After this first resume, new
periodic checkpoints land in `checkpoint_dir` and a plain re-run (without
`resume_from_checkpoint`) would auto-detect those instead.

**To persist past this session**: click "Save Version" when done (or
periodically) -- that's Kaggle's equivalent of Drive surviving a disconnect.
The saved output can be attached to a new session the same way the epoch-24
checkpoint was.

In [ ]:
from src.train import main as train_main

train_main(
    dataset_root=MANIFEST_DIR,
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path="/kaggle/working/meshgraphnet.ckpt",
    max_epochs=100,
    batch_size=1,
    accumulate_grad_batches=4,
    n_val=80,
    checkpoint_every_n_epochs=5,
    num_workers=2,
    precision="16-mixed",
    resume_from_checkpoint=RESUME_CHECKPOINT,
)